In [ ]:
import os
import pandas as pd

# Define the path to the downloads folder for user TrevorWhite
downloads_folder = r"C:\Users\TrevorWhite\Downloads"

# List of specific file names
file_names = [
    "MW_PBP_24.csv",
    "WCC_PBP_24.csv",
    "SEC_PBP_24.csv",
    "P12_PBP_24.csv",
    "B10_PBP_24.csv",
    "BE_PBP_24.csv",
    "B12_PBP_24.csv",
    "ACC_PBP_24.csv"
]

# List to store data from each file
data_list = []

# Read and append each file
for file_name in file_names:
    file_path = os.path.join(downloads_folder, file_name)
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        data_list.append(df)
        print(f"Successfully read: {file_name}")
    else:
        print(f"File not found: {file_name}")

# Combine all data into a single DataFrame
combined_df = pd.concat(data_list, ignore_index=True)
print("All files successfully appended into a single DataFrame.")


In [ ]:
# List of columns to select
columns_to_select = [
    "uniqPitchId",
    "pitcherId",
    "Spin",
    "Extension",
    "PitchType",
    "HorzApprAngle",
    "VertApprAngle",
    "HorzRelAngle",
    "VertRelAngle",
    "IndVertBrk",
    "HorzBrk",
    "RelZ",
    "RelX",
    "Vel",
    "pitchResult",
    "count",
    "PZ",
    "PX"
]

# Select the desired columns from the combined DataFrame
selected_columns_df = combined_df[columns_to_select]
print("Selected relevant columns.")


In [ ]:
import numpy as np

def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Feature engineering for a baseball dataset.
    """

    # Ensure relevant columns are numeric
    numeric_columns = ["Spin",
                        "Extension",
                        "HorzApprAngle",
                        "VertApprAngle",
                        "HorzRelAngle",
                        "VertRelAngle",
                        "IndVertBrk",
                        "HorzBrk",
                        "RelZ",
                        "RelX",
                        "Vel",
                        "PZ",
                        "PX"    ]
    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=numeric_columns)  # Drop rows with missing numeric data

    # Determine pitcher handedness
    df_hand = (
        df.groupby("pitcherId", as_index=False)["RelX"].mean()
        .rename(columns={"RelX": "avg_RelX"})
    )
    df_hand["pitcher_hand"] = np.where(df_hand["avg_RelX"] > 0, "R", "L")
    df = pd.merge(df, df_hand[["pitcherId", "pitcher_hand"]], on="pitcherId", how="left")

    # Rename columns to standard references
    df = df.rename(columns={
        "Vel": "start_speed",
        "Spin": "spin_rate",
        "Extension": "extension",
        "RelZ": "z0",
        "RelX": "x0",
        "HorzBrk": "ax",
        "IndVertBrk": "az",
        "PitchType": "pitch_type"
    })

    # Mirror for left-handed pitchers
    df["ax"] = np.where(df["pitcher_hand"] == "L", -df["ax"], df["ax"])
    df["x0"] = np.where(df["pitcher_hand"] == "L", -df["x0"], df["x0"])

    # Most-used fastball logic
    fastball_types = ["FF", "SI", "FA"]
    df_fb = df[df["pitch_type"].isin(fastball_types)].copy()
    df_agg = (
        df_fb.groupby(["pitcherId", "pitch_type"], as_index=False)
        .agg(
            avg_fastball_speed=("start_speed", "mean"),
            avg_fastball_az=("az", "mean"),
            avg_fastball_ax=("ax", "mean"),
            count=("start_speed", "count")
        )
    )
    df_agg = df_agg.sort_values(["count", "avg_fastball_speed"], ascending=[False, False])
    df_agg = df_agg.drop_duplicates(subset=["pitcherId"], keep="first")
    df = pd.merge(
        df,
        df_agg[["pitcherId", "avg_fastball_speed", "avg_fastball_az", "avg_fastball_ax"]],
        on="pitcherId",
        how="left"
    )

    df["speed_diff"] = df["start_speed"] - df["avg_fastball_speed"]
    df["az_diff"] = df["az"] - df["avg_fastball_az"]
    df["ax_diff"] = df["ax"] - df["avg_fastball_ax"]

    return df


In [ ]:
# Step 1: Assign 0 for cases where pitchResult contains "walk", "looking", or "ball in the dirt"
selected_columns_df["is_swing"] = selected_columns_df["pitchResult"].apply(
    lambda x: 0 if any(keyword in x.lower() for keyword in ["walk", "looking", "ball in the dirt"]) else 1
)

# Step 2: Assign 0 for rows where pitchResult equals "ball" (exact match)
selected_columns_df.loc[
    selected_columns_df["pitchResult"].str.lower() == "ball", "is_swing"
] = 0

# Step 3: Assign 1 to is_whiff for swinging strikes
# Assign 1 to is_whiff where pitchResult contains "swinging" (case-insensitive)
selected_columns_df["is_whiff"] = selected_columns_df["pitchResult"].apply(
    lambda x: 1 if "swinging" in x.lower() else 0
)


In [ ]:
# Filter out waste pitches, "Hit By Pitch", "bunt", and "intentional"
selected_columns_df = selected_columns_df[
    ~(
        (selected_columns_df["PZ"] > 5) |
        (selected_columns_df["PZ"] < 0) |
        (selected_columns_df["PX"] > 2.5) |
        (selected_columns_df["PX"] < -2.5) |
        (selected_columns_df["pitchResult"].str.contains("interference", case=False)) |
        (selected_columns_df["pitchResult"].str.contains("hit by pitch", case=False)) |
        (selected_columns_df["pitchResult"].str.contains("bunt", case=False)) |
        (selected_columns_df["pitchResult"].str.contains("intentional", case=False)) |
        (selected_columns_df["pitchResult"].str.contains("unknown", case=False))
    )
]


In [ ]:
# Filter rows where is_swing is 1
swing_results = selected_columns_df[selected_columns_df["is_swing"] == 1]["pitchResult"].unique()

# Filter rows where is_whiff is 1
whiff_results = selected_columns_df[selected_columns_df["is_whiff"] == 1]["pitchResult"].unique()

# Display the results
print("PitchResults for is_swing == 1:")
print(swing_results)

print("\nPitchResults for is_whiff == 1:")
print(whiff_results)


In [ ]:
# Filter rows where is_swing is 1
nswing_results = selected_columns_df[selected_columns_df["is_swing"] == 0]["pitchResult"].unique()

# Filter rows where is_whiff is 1
# Display the results
print("PitchResults for is_swing == 1:")
print(nswing_results)

In [ ]:
# Apply feature engineering to the selected columns DataFrame
df_engineered = feature_engineering(selected_columns_df)

# Display the first few rows of the engineered DataFrame to inspect the changes
print(df_engineered.head())


In [ ]:
# Step 1: Add Boolean Column for Fastballs
df_engineered["is_fastball"] = df_engineered["pitch_type"].isin(["FF", "FA", "SI"])



# Filter for only swings
df_engineered = df_engineered[df_engineered["is_swing"] == 1]



In [ ]:
# Define the features to be used for training
features = [
    "start_speed",
    "spin_rate",
    "extension",
    "az",
    "ax",
    "x0",
    "z0",
    "speed_diff",
    "az_diff",
    "ax_diff",
    "is_fastball"
]

# Define the target variable
target = "is_whiff"  # Replace "is_whiff" with the desired target column if needed


In [ ]:
# Step 1: Define the columns to select (pitch_type + features + target)
columns_to_select = ["pitch_type", "pitcherId"] + features + [target]

# Step 2: Select the defined columns from df_engineered
df_selected = df_engineered[columns_to_select]

# Step 3: Group by 'pitch_type' and 'pitcherId', calculate mean and count
df_grouped = (
    df_selected
    .groupby(["pitch_type", "pitcherId"], as_index=False)
    .agg({**{col: "mean" for col in features + [target]}, "pitch_type": "count"})
    .rename(columns={"pitch_type": "count"})
)

# Display the first few rows of the grouped DataFrame
df_grouped.head(111)




In [ ]:
# Total length of df_grouped
total_length = len(df_grouped)

# Number of observations where count > 3
count_greater_than_3 = len(df_grouped[df_grouped["count"] > 4])

# Display the results
print(f"Total number of groups: {total_length}")
print(f"Number of groups with count > 3: {count_greater_than_3}")


In [ ]:
# Drop rows with null values in the specified features and target column
df_train = df_engineered.dropna(subset=features + [target])


# Display the shape of the training DataFrame to verify
df_train.head()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import joblib
import matplotlib.pyplot as plt

# Split data into features (X) and target (y)
X = df_train[features]
y = df_train[target]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
# Define and train the model pipeline
model = make_pipeline(
    StandardScaler(),
    RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=10,
        random_state=42
    )
)

# Fit the model
model.fit(X_train, y_train)

# Make predictions
y_pred_proba = model.predict_proba(X_test)[:, 1]  # Probability of is_whiff (label 1)
y_pred = model.predict(X_test)  # Predicted labels

# Evaluate the model
roc_auc = roc_auc_score(y_test, y_pred_proba)
accuracy = accuracy_score(y_test, y_pred)

print(f"ROC AUC Score: {roc_auc:.3f}")
print(f"Accuracy: {accuracy:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import joblib
import matplotlib.pyplot as plt

# Filter rows with count > 3
df_grouped = df_grouped[df_grouped["count"] > 3]

# Subset to features and target
df_train_group = df_grouped[features + [target]]

# Split data into features (X) and target (y)
X_group = df_train_group[features]
y_group = df_train_group[target]

# Split data into training and testing sets
X_train_group, X_test_group, y_train_group, y_test_group = train_test_split(
    X_group, y_group, test_size=0.2, random_state=42
)


In [ ]:
# Define and train the model pipeline
group_model = make_pipeline(
    StandardScaler(),
    RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        min_samples_split=10,
        random_state=42
    )
)

# Fit the model
group_model.fit(X_train_group, y_train_group)

# Make predictions
y_pred_group = group_model.predict(X_test_group)

# Evaluate the model
mse = mean_squared_error(y_test_group, y_pred_group)
r2 = r2_score(y_test_group, y_pred_group)

print(f"Mean Squared Error (MSE): {mse:.3f}")
print(f"R² Score: {r2:.3f}")


In [ ]:
# Save the model
joblib.dump(group_model, "C:/Users/TrevorWhite/Downloads/whiff_model_grouped_training.joblib")
print("Model saved to whiff_model_grouped.joblib")

# Plot feature importances
rf_group_model = group_model.named_steps["randomforestregressor"]
feature_importances_group = rf_group_model.feature_importances_

plt.figure(figsize=(10, 6))
plt.barh(X_group.columns, feature_importances_group, color="skyblue")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Feature Importances (Grouped Data)")
plt.gca().invert_yaxis()
plt.show()
